<div style="width: 100%; overflow: hidden;">
    <a href="http://www.uc.pt/fctuc/dei/">
    <div style="float:left; width: 75%;">
        <img src="https://eden.dei.uc.pt/~naml/images_ecos/dei25.png"/>
    </div>
    </a>
</div>

# Introduction to Machine Learning - Day 2
## A 2.5-hour hands-on session - Breast Cancer Wisconsin Diagnostic Database

---

### Where we left off

Day 1 got you from zero to four working classifiers, and ended right at
**Challenge 1**, where you searched for the best pair of measurements out of
30. Today picks up from exactly there, on the exact same dataset. Nothing new
to load, nothing new to install.

### What today adds

Day 1 answered "can we build a model that works?". Today answers the harder
question: **can we trust the number it gives us?** We will find three separate
ways a model can lie to you - one split getting lucky, a leaked answer hiding
in the data, and a great score that means something different depending on
who you test it on - and then we will build a neural network and see whether
it actually helps.

### How today works

Same loop as Day 1, same rhythm:

| step | what it is | what you do |
|---|---|---|
| **1. Concept** | one idea, explained in a few lines | read it |
| **2. Code Breakdown** | the new code, line by line | read it |
| **3. Code cell** | a working cell with one `# TODO` | run it, then change the TODO and run again |
| **4. Chat check-in** | one line to post in the chat | post it, we wait for about half the room |

Today's TODOs occasionally ask for a full line of code instead of one number -
you have the basics now, so we lean on them a little more.

---
---
# PART 0 - Picking up where we stopped
### 0:00 - 0:10

Run the next cell once. It repeats, unchanged, everything Day 1 did before
Challenge 1: load the data, split it, and fit a logistic regression. If
anything here looks unfamiliar, that is exactly what Day 1 covered - ask in
the chat before we move on.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

# Load the data, exactly like Day 1
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)   # 569 patients, 30 measurements
y = pd.Series(data.target)                                 # 0 = malignant, 1 = benign

# Split into train and test, exactly like Day 1
train, test, train_labels, test_labels = train_test_split(
    X, y, test_size=0.33, random_state=RANDOM_STATE, stratify=y)

# Fit the same first model as Day 1
logreg = LogisticRegression(max_iter=5000)
logreg.fit(train, train_labels)
preds = logreg.predict(test)

print("Training patients:", len(train), " Test patients:", len(test))
print(f"Logistic regression accuracy: {accuracy_score(test_labels, preds):.4f}")

**One change from Day 1, worth noticing.** We added `stratify=y` to the split.
It forces both the training set and the test set to keep the same proportion
of malignant and benign patients as the full dataset. Without it, a run of bad
luck could put too many of one class in the test set, and every number that
follows would be noise. Small change, and from here on we always use it.

---
---
# BLOCK 1 - Can we trust one number?
### 0:10 - 0:40

Every score in Day 1 came from a single split: one particular 33% of patients
held back as the test set. Block 1 asks the uncomfortable question - what if
we had split it differently?

## Task 1 of 11 - One split, many answers

### 1. Concept

`random_state` controls exactly which patients land in the test set. Change it
and you get a different, equally valid 33%. If the model is genuinely good,
the accuracy should barely move when we do that. If it swings a lot, the
single number Day 1 reported was partly luck.

### 2. Code Breakdown

A loop we already know from Day 1, run once per seed:

```python
for seed in seeds:
    tr, te, trl, tel = train_test_split(X, y, test_size=0.33,
                                         random_state=seed, stratify=y)
    m = LogisticRegression(max_iter=5000)
    m.fit(tr, trl)
    accs.append(accuracy_score(tel, m.predict(te)))
```

`accs` collects one accuracy per seed. At the end, `np.std(accs)` measures how
much they disagree - close to 0 means "very consistent", larger means "this
score depends heavily on which patients happened to be held back".

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: Add three more seeds to the list, e.g. [0, 1, 2, 3, 4, 5]
seeds = [0, 1, 2]

accs = []
for seed in seeds:
    tr, te, trl, tel = train_test_split(X, y, test_size=0.33,
                                         random_state=seed, stratify=y)
    m = LogisticRegression(max_iter=5000)
    m.fit(tr, trl)
    accs.append(accuracy_score(tel, m.predict(te)))

print("Accuracy per seed:", [round(a, 4) for a in accs])
print(f"Lowest: {min(accs):.4f}   Highest: {max(accs):.4f}   Spread: {np.std(accs):.4f}")

---
**CHAT CHECK-IN**

1. Run the cell above (**Shift + Enter**).
2. Change `seeds` to `[0, 1, 2, 3, 4, 5]` and run again.
3. Post in the chat: `lowest = <n>, highest = <n>`

We move on once about half the chat has answered. If you got red text instead, paste the **last line** of it.

---

## Task 2 of 11 - Cross-validation: five tests instead of one

### 1. Concept

**Cross-validation** stops us picking just one split. It cuts the training
data into `n_splits` equal slices, trains on all but one, tests on the slice
left out, and repeats until every slice has had a turn as the test set. You get
several scores from data you never doubled up, not one.

### 2. Code Breakdown

```python
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(logreg, train, train_labels, cv=cv, scoring="roc_auc")
```

`StratifiedKFold` is the slicing plan - "stratified" means every slice keeps
the same malignant/benign balance, same idea as `stratify=y` above.
`cross_val_score` does the training and testing for you, `n_splits` times, and
hands back one score per slice.

`scoring="roc_auc"` asks for a different score than accuracy - **AUC**, which
we will meet properly in Block 3. For now, read it exactly like accuracy: 1.0
is perfect, 0.5 is a coin flip.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# TODO: Change n_splits from 5 to 10
n_splits = 5

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
logreg = LogisticRegression(max_iter=5000)
scores = cross_val_score(logreg, train, train_labels, cv=cv, scoring="roc_auc")

print(f"n_splits = {n_splits}")
print("Score per fold:", scores.round(3))
print(f"Average AUC: {scores.mean():.4f}   Spread (std): {scores.std():.4f}")

---
**CHAT CHECK-IN**

1. Run the cell above (**Shift + Enter**).
2. Note the average AND the spread - both matter.
3. Post in the chat: `average = <n>, spread = <n>`

We move on once about half the chat has answered. If you got red text instead, paste the **last line** of it.

---

**What to notice.** The spread is small here - this dataset is unusually
well-behaved. On messier data, a spread of 0.05 or more is common, and it is
the honest half of the result: an average of 0.95 with a spread of 0.05 is a
far weaker claim than the average on its own suggests. Always report both.

## Task 3 of 11 - Comparing two models fairly

### 1. Concept

Day 1's comparison table ranked four models using one test-set score each.
Task 1 just showed that one score can move around by chance. Cross-validation
gives every model the same fair fight: the same five slices, so the comparison
is not distorted by one model getting an easier split than another.

### 2. Code Breakdown

The same dictionary trick as Day 1's comparison table, but scored with
`cross_val_score` instead of a single `.fit()` / `.predict()`:

```python
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Naive Bayes": GaussianNB(),
}

for name, m in models.items():
    scores = cross_val_score(m, train, train_labels, cv=cv, scoring="roc_auc")
    print(f"{name:<20} average AUC = {scores.mean():.4f}  (spread {scores.std():.4f})")
```

**Your turn to add a line.** A dictionary entry has the same shape as the ones
already there: `"Name": Model(settings)`.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Naive Bayes": GaussianNB(),
    # TODO: Add a third model here, e.g. "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, m in models.items():
    scores = cross_val_score(m, train, train_labels, cv=cv, scoring="roc_auc")
    print(f"{name:<22} average AUC = {scores.mean():.4f}  (spread {scores.std():.4f})")

---
**CHAT CHECK-IN**

1. Add a `"Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)` line to the dictionary and run the cell.
2. Post in the chat: `best model = <name>, average AUC = <n>`

We move on once about half the chat has answered. If you got red text instead, paste the **last line** of it.

---

---
---
# BLOCK 2 - Data leakage: the planted clue
### 0:40 - 1:05

The most common way a real clinical model fails is not a bad algorithm. It is
a column that, without anyone noticing, already contains the answer.

## Task 4 of 11 - A suspiciously perfect model

### 1. Concept

Imagine a hospital's spreadsheet has one more column: a note a radiologist
added, but only *after* the biopsy result came back confirming cancer. Feed
that column to a model and it will look brilliant - because it is not
predicting the diagnosis, it is reading it.

Below we build exactly that column ourselves, on purpose, so we can watch it
happen. `lab_note_score` is built from the real answer (`train_labels`) plus
some random noise. `NOISE` controls how well-hidden the leak is - `0.01` means
the answer is barely disguised at all.

### 2. Code Breakdown

```python
rng = np.random.RandomState(0)
leak_column = train_labels.values + rng.normal(0, NOISE, size=len(train_labels))
```

We attach `leak_column` as `lab_note_score` and train a model on **that single
column alone** - no real measurements at all - to see how far a leaked answer
can carry a model by itself.

In [ ]:
# TODO: Change NOISE from 0.01 to 0.3
NOISE = 0.01

rng = np.random.RandomState(0)
leak_column = train_labels.values + rng.normal(0, NOISE, size=len(train_labels))

leaky_data = pd.DataFrame({"lab_note_score": leak_column})

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
leak_score = cross_val_score(DecisionTreeClassifier(random_state=RANDOM_STATE),
                              leaky_data, train_labels, cv=cv, scoring="roc_auc").mean()

real_score = cross_val_score(DecisionTreeClassifier(random_state=RANDOM_STATE),
                              train, train_labels, cv=cv, scoring="roc_auc").mean()

print(f"NOISE = {NOISE}")
print(f"AUC using ONLY the leaked column         : {leak_score:.4f}")
print(f"AUC using all 30 real measurements        : {real_score:.4f}")

---
**CHAT CHECK-IN**

1. Run the cell above (**Shift + Enter**).
2. Change `NOISE` to `0.3` and run again - the leak is now noisier, but watch what still happens.
3. Post in the chat: `noise=0.3 leak AUC = <n>, real AUC = <n>`

We move on once about half the chat has answered. If you got red text instead, paste the **last line** of it.

---

**What to notice.** Even heavily disguised, one leaked column beats thirty
honest measurements. In a hospital database with hundreds of columns this is
never this obvious - nobody names a column `lab_note_score`. The rule that
catches it: for every column, ask **when was this recorded, relative to the
diagnosis?** If the answer is "after it" or "because of it", it does not
belong in the model, however good it makes the score look.

## Task 5 of 11 - Building the pipeline that cannot leak by accident

### 1. Concept

The leak above was obvious because we planted it. A subtler version of the
same mistake: computing something from the *whole* dataset - a scaling factor,
a selected set of features - before splitting into train and test. The test
set then quietly influences a decision that was supposed to be made blind to
it.

A `Pipeline` fixes this by construction. It bundles preprocessing and the
model into one object, so every step is refit from scratch on whichever
patients are currently the "training" ones - automatically correct, every
time, including inside `cross_val_score`.

### 2. Code Breakdown

```python
pipe = Pipeline([
    ("scale", StandardScaler()),          # step 1: put every measurement on a common scale
    ("model", LogisticRegression(max_iter=5000)),   # step 2: fit on the scaled version
])
```

Calling `cross_val_score(pipe, ...)` now refits `StandardScaler` freshly
inside each of the five folds - the scaling never sees the fold being held
out. We will need this exact pattern again in Block 4 for the neural network.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# TODO: Change n_splits from 5 to 3
n_splits = 5

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000)),
])

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
pipe_scores = cross_val_score(pipe, train, train_labels, cv=cv, scoring="roc_auc")

print(f"n_splits = {n_splits}")
print(f"Pipeline (scale + model) average AUC: {pipe_scores.mean():.4f}  (spread {pipe_scores.std():.4f})")

---
**CHAT CHECK-IN**

1. Run the cell above (**Shift + Enter**).
2. Post in the chat: `pipeline AUC = <n>`

We move on once about half the chat has answered. If you got red text instead, paste the **last line** of it.

---

---
### Break - 10 minutes

Leave the screen. When we come back: ROC curves, and the question that
matters more than any accuracy number - **how common is the disease in the
people you actually test?**

---
---
# BLOCK 3 - ROC, AUC, and the prevalence trap
### 1:15 - 1:45

Day 1's Challenge 2 asked you to move a threshold and watch missed cancers and
false alarms trade off. Block 3 gives you the tool that shows every possible
threshold at once, and then a much bigger surprise: the same model, the same
threshold, can look excellent or useless depending only on **who you point it
at**.

## Task 6 of 11 - Probabilities instead of yes/no

### 1. Concept

`.predict()` hides information: it silently turns "82% confident" into a flat
"benign". `.predict_proba()` returns the confidence itself, for both classes,
as two columns.

In this dataset, column **0** is the probability of **malignant** and column
**1** is the probability of **benign** - `data.target_names` from Day 1 gives
that order. Since catching cancer is what matters clinically, from here on we
read off **column 0**.

### 2. Code Breakdown

```python
proba_malignant = pipe.predict_proba(test)[:, 0]   # confidence of cancer, per patient
```

`proba_malignant` is one number per test patient, between 0 and 1, instead of
one word.

In [ ]:
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000)),
])
pipe.fit(train, train_labels)

# TODO: Change how_many from 5 to 10
how_many = 5

proba_malignant = pipe.predict_proba(test)[:, 0]

print("Probability of malignant, first patients:", proba_malignant[:how_many].round(3))
print("What .predict() collapsed those into    :", pipe.predict(test)[:how_many])
print("0 = malignant, 1 = benign")

---
**CHAT CHECK-IN**

1. Run the cell above (**Shift + Enter**).
2. Find a probability close to 0.5 among the first ten - that patient is the model's most uncertain call.
3. Post in the chat: `closest to 0.5 = <n>`

We move on once about half the chat has answered. If you got red text instead, paste the **last line** of it.

---

## Task 7 of 11 - The ROC curve and AUC

### 1. Concept

Day 1's Challenge 2 tried one threshold at a time by hand. The **ROC curve**
plots every possible threshold in one picture: for each one, how many cancers
it catches (**sensitivity**, y-axis) against how many false alarms it raises
(1 - **specificity**, x-axis). A curve that hugs the top-left corner is a model
that can catch cancers without raising many false alarms.

**AUC** (area under that curve) compresses the whole picture into one number:
the probability that a randomly chosen malignant patient gets a higher risk
score than a randomly chosen benign one. 1.0 is perfect ordering, 0.5 is
random guessing.

### 2. Code Breakdown

```python
fpr, tpr, thresholds = roc_curve(test_is_malignant, proba_malignant)
auc = roc_auc_score(test_is_malignant, proba_malignant)
```

`test_is_malignant` has to be 1 for malignant and 0 for benign - the opposite
of `test_labels` - because `roc_curve` needs to know which class we are trying
to catch.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

test_is_malignant = (test_labels == 0).astype(int)

fpr, tpr, thresholds = roc_curve(test_is_malignant, proba_malignant)
auc = roc_auc_score(test_is_malignant, proba_malignant)

# TODO: Change color from "#c44e52" to "#4c72b0"
color = "#c44e52"

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color=color, linewidth=3, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="random guessing")
plt.xlabel("False alarm rate (1 - specificity)")
plt.ylabel("Cancers caught (sensitivity)")
plt.title("ROC curve - full 30-feature model")
plt.legend()
plt.show()

print(f"AUC = {auc:.4f}")

---
**CHAT CHECK-IN**

1. Run the cell above (**Shift + Enter**).
2. Look at how close the curve gets to the top-left corner.
3. Post in the chat: `AUC = <n>`

We move on once about half the chat has answered. If you got red text instead, paste the **last line** of it.

---

## Task 8 of 11 - The prevalence trap

### 1. Concept

Here is the uncomfortable part. In this dataset, about **37%** of patients are
malignant - because the dataset was built that way, one case gathered for
roughly every couple of controls. That is not how disease works in a
population you would actually screen.

**PPV** (positive predictive value) answers the only question a real patient
cares about: *"the test came back positive - what are the actual odds I have
cancer?"* PPV depends on sensitivity and specificity, but it depends **just as
much** on how common the disease is in whoever you are testing. Sensitivity
and specificity themselves do not change with prevalence - PPV does, sharply.

We deliberately reuse the **cheap 2-feature model** from Day 1's Challenge 1:
a screening test cheap enough to run on everyone gives more false alarms than
the full 30-feature model, which makes the trap far easier to see.

### 2. Code Breakdown

```python
def ppv(sensitivity, specificity, prevalence):
    true_positives  = sensitivity * prevalence
    false_positives = (1 - specificity) * (1 - prevalence)
    return true_positives / (true_positives + false_positives)
```

We fix sensitivity and specificity at the values from the cheap model's ROC
curve, then change only `prevalence` - nothing about the model itself moves.

In [ ]:
cheap_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000)),
])
cheap_pipe.fit(train[["mean radius", "mean texture"]], train_labels)
proba_cheap = cheap_pipe.predict_proba(test[["mean radius", "mean texture"]])[:, 0]

fpr_c, tpr_c, thr_c = roc_curve(test_is_malignant, proba_cheap)
i = np.argmin(np.abs(tpr_c - 0.95))          # the threshold that catches ~95% of cancers
sens, spec = tpr_c[i], 1 - fpr_c[i]
print(f"Cheap 2-feature model at this threshold: sensitivity = {sens:.3f}, specificity = {spec:.3f}")

def ppv(sensitivity, specificity, prevalence):
    true_positives = sensitivity * prevalence
    false_positives = (1 - specificity) * (1 - prevalence)
    return true_positives / (true_positives + false_positives)

# TODO: Add 0.02 to the list of prevalences
prevalences = [0.37, 0.10]

rows = []
for prevalence in prevalences:
    value = ppv(sens, spec, prevalence)
    rows.append({"cancer rate in population": f"{prevalence:.1%}",
                 "chance a positive result is real": f"{value:.1%}",
                 "false alarms per real cancer found": f"{(1 - value) / value:.1f}"})
print(pd.DataFrame(rows).to_string(index=False))

---
**CHAT CHECK-IN**

1. Add `0.02` to `prevalences` (a realistic screening population) and run again.
2. Post in the chat: `at 2% prevalence, PPV = <n>, false alarms per case = <n>`

We move on once about half the chat has answered. If you got red text instead, paste the **last line** of it.

---

**What to notice.** The model did not change. Sensitivity and specificity did
not change. Only the population changed, and the test collapsed from
"clinically useful" to "mostly false alarms". This is why a paper reporting
only an AUC on a balanced research dataset tells you far less than it
appears to about deploying that model in the real world.

---
---
# CHALLENGE - The Screening Rollout
### 1:45 - 2:00 (15 minutes)

### The scenario

The hospital wants to deploy the cheap 2-feature test as a **first-line
screen** offered to every woman attending a routine check-up. In that
population, roughly **2 in 100** actually have a malignant tumour - nothing
like the 37% in our dataset.

Your job: choose a **sensitivity target** (how many cancers you insist on
catching) and see what that decision costs in false alarms, once it is
deployed at real-world prevalence.

### How to do it

1. Run the cell below unchanged once.
2. Change `SENS_TARGET` - try `0.90`, `0.95`, `0.99` - and watch specificity,
   PPV, and the false-alarm count all move together.

In [ ]:
# =====================================================================
#  CHALLENGE - change SENS_TARGET below, then run the cell.
# =====================================================================

# TODO: Change SENS_TARGET from 0.95 to 0.99
SENS_TARGET = 0.95

PREVALENCE = 0.02   # 2 in 100, a realistic screening population

i = np.argmin(np.abs(tpr_c - SENS_TARGET))
sens, spec = tpr_c[i], 1 - fpr_c[i]
value = ppv(sens, spec, PREVALENCE)

print(f"Target sensitivity : {SENS_TARGET:.2f}")
print(f"Achieved sensitivity: {sens:.3f}   Achieved specificity: {spec:.3f}")
print(f"At {PREVALENCE:.0%} prevalence:")
print(f"  Chance a positive result is real cancer: {value:.1%}")
print(f"  False alarms sent for further testing, per real cancer found: {(1 - value) / value:.1f}")

---
**CHAT CHECK-IN - Challenge leaderboard**

Post in this exact format:

`sens_target=<t>: achieved sens=<n>, PPV=<n>, false alarms per case=<n>`

Example: `sens_target=0.99: achieved sens=0.986, PPV=0.041, false alarms per case=23.4`

---

### Discussion - two minutes

Pushing sensitivity to 0.99 does not make the test better for free - it buys
extra caught cancers with a steep rise in false alarms, exactly like Day 1's
Challenge 2. The difference today is that the price is being paid **at 2%
prevalence**, which is what a real screening rollout looks like, not at the
37% this dataset happens to contain.

---
---
# BLOCK 4 - Neural networks
### 2:00 - 2:30

Four algorithms so far all draw one kind of boundary between malignant and
benign. A **neural network** builds its boundary out of many small boundaries
stacked on top of each other. Let's see whether that extra machinery earns
its keep here.

## Task 9 of 11 - What a neural network actually is

### 1. Concept

Look back at logistic regression: it multiplies every measurement by a
learned weight, adds them up, and squeezes the result into a probability
between 0 and 1. That is, in fact, **one neuron**.

A neural network stacks many of those in a row (a **layer**), then feeds every
output of that layer into a fresh row of them (another layer), and so on,
before a final neuron produces the probability. Each stack is called a
**hidden layer**; the neurons inside are its **width**. More layers, or wider
ones, let the network combine the 30 measurements in more complicated ways -
at the cost of needing more data to learn all those extra weights without
memorising noise.

### 2. Code Breakdown

Same three-step shape you already know, one new class:

```python
mlp = MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000, random_state=RANDOM_STATE)
```

`hidden_layer_sizes=(16,)` means **one hidden layer of 16 neurons**.
`(64, 32)` would mean two layers, of 64 then 32 neurons. `max_iter` is the
neural-network version of Day 1's `max_iter` for logistic regression - the
number of passes it is allowed over the training data before giving up.

**Neural networks are sensitive to scale** in a way trees are not - a
measurement in the hundreds can dominate one in the tenths for no good reason.
That is exactly what the `Pipeline` from Task 5 is for, and we reuse it
unchanged.

In [ ]:
from sklearn.neural_network import MLPClassifier

# TODO: Change hidden_layer_sizes from (16,) to (64, 32)
hidden_layer_sizes = (16,)

mlp_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", MLPClassifier(hidden_layer_sizes=hidden_layer_sizes,
                             max_iter=2000, random_state=RANDOM_STATE)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
mlp_scores = cross_val_score(mlp_pipe, train, train_labels, cv=cv, scoring="roc_auc")

print(f"hidden_layer_sizes = {hidden_layer_sizes}")
print(f"Neural network average AUC: {mlp_scores.mean():.4f}  (spread {mlp_scores.std():.4f})")

---
**CHAT CHECK-IN**

1. Run the cell above (**Shift + Enter**).
2. Change `hidden_layer_sizes` to `(64, 32)` and run again - two layers instead of one.
3. Post in the chat: `(16,) AUC = <n>, (64, 32) AUC = <n>`

We move on once about half the chat has answered. If you got red text instead, paste the **last line** of it.

---

## Task 10 of 11 - Does scaling actually matter here?

### 1. Concept

Task 9 said neural networks are sensitive to scale. Let's not take that on
faith - fit the same network with and without the `StandardScaler` step and
compare, the same way Day 1 tested claims by changing one thing at a time.

### 2. Code Breakdown

```python
mlp_unscaled = MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000, random_state=RANDOM_STATE)
```

No `Pipeline`, no `StandardScaler` - the raw 30 measurements, on their
original scales, go straight into the network.

In [ ]:
# TODO: Change max_iter from 2000 to 200
max_iter = 2000

mlp_unscaled = MLPClassifier(hidden_layer_sizes=(16,), max_iter=max_iter, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
unscaled_scores = cross_val_score(mlp_unscaled, train, train_labels, cv=cv, scoring="roc_auc")

print(f"max_iter = {max_iter}")
print(f"Without scaling, average AUC: {unscaled_scores.mean():.4f}")
print(f"With scaling (Task 9, (16,)): compare against your answer above")

---
**CHAT CHECK-IN**

1. Run the cell above (**Shift + Enter**).
2. Change `max_iter` to `200` and run again - fewer passes over the data.
3. Post in the chat: `unscaled AUC = <n>, at max_iter=200 AUC = <n>`

We move on once about half the chat has answered. If you got red text instead, paste the **last line** of it.

---

**What to notice.** The gap from skipping scaling is real but modest here,
because this dataset is small and clean. On larger, messier tabular data the
same mistake can cost far more - the habit of always scaling before a neural
network is worth keeping regardless of how forgiving today's data happens to
be.

## Task 11 of 11 - The full showdown

### 1. Concept

One table, every model from both days, scored the fair way - cross-validation,
same five folds, same metric. This is the table Day 1's version should have
been.

### 2. Code Breakdown

Exactly Task 3's dictionary pattern, extended.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# TODO: Add "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5) to the dictionary
from sklearn.neighbors import KNeighborsClassifier

models = {
    "Logistic Regression": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=5000))]),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
    "Gradient Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    "Neural Network": Pipeline([("scale", StandardScaler()),
                                 ("model", MLPClassifier(hidden_layer_sizes=(32,), max_iter=2000, random_state=RANDOM_STATE))]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, m in models.items():
    scores = cross_val_score(m, train, train_labels, cv=cv, scoring="roc_auc")
    rows.append({"Model": name, "Average AUC": round(scores.mean(), 4), "Spread": round(scores.std(), 4)})

results = pd.DataFrame(rows).sort_values("Average AUC", ascending=False)
print(results.to_string(index=False))

---
**CHAT CHECK-IN**

1. Add the K-Nearest Neighbors line to the dictionary and run the cell.
2. Post in the chat: `best = <model>, AUC = <n>, worst = <model>, AUC = <n>`

We move on once about half the chat has answered. If you got red text instead, paste the **last line** of it.

---

**What to notice.** The differences between the top few models are almost
always smaller than their spreads. On 569 patients and 30 clean measurements,
the neural network has nothing to be deep about - it typically ties the
simpler models rather than beating them. That is not a failure of neural
networks; it is a statement about when they are the right tool. They tend to
pull ahead on problems with far more data, or far messier raw inputs
(images, text, audio) than a tidy table of 30 numbers.

---
### Optional, if time allows - the same network in Keras

Everything above used `MLPClassifier`, which is scikit-learn's own neural
network - three lines, same shape as everything else today. The heavier,
industry-standard way to build one is **TensorFlow / Keras**, which is what
powers image and language models at a much larger scale. It needs a separate
install and a minute to load, so treat this as a look, not a requirement.

```python
# Optional - only run this if TensorFlow is available in your environment.
# !pip install -q tensorflow

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(32, activation="relu", input_shape=(train.shape[1],)),  # hidden layer, 32 neurons
    layers.Dense(16, activation="relu"),                                  # second hidden layer, 16 neurons
    layers.Dense(1,  activation="sigmoid"),                               # output: one probability
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["AUC"])

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(train)

model.fit(scaler.transform(train), 1 - train_labels,   # 1 = malignant, to match Block 3
          epochs=30, batch_size=16, verbose=0)

test_auc = model.evaluate(scaler.transform(test), 1 - test_labels, verbose=0)[1]
print(f"Keras neural network AUC: {test_auc:.4f}")
```

Read it against Task 9's `MLPClassifier`: `Dense(32, activation="relu")` is
one hidden layer of 32 neurons, the same idea as `hidden_layer_sizes=(32,)`.
`.compile()` has no equivalent in scikit-learn - Keras makes you say explicitly
how the network should measure its own mistakes (`loss`) and how it should
correct them (`optimizer`), decisions scikit-learn made for you by default.
`epochs` is `max_iter` under a different name. Expect a very similar AUC to
the `MLPClassifier` version - same idea, different library, same lesson from
Task 11 about when the extra machinery pays off.

---
---
# Wrap-up

### The three ways a model can lie to you

1. **One split can be lucky or unlucky.** Cross-validation replaces one score
   with several, and the spread between them tells you how much to trust the
   average.
2. **A column that already knows the answer will make any model look
   brilliant.** Before trusting a feature, ask when it was recorded relative
   to the outcome. Wrap preprocessing in a `Pipeline` so the test data never
   quietly influences a decision made "before" the split.
3. **A great AUC on a balanced research dataset can still be a useless test
   in the real world.** PPV - not sensitivity, not specificity, not AUC -
   is the number a patient actually needs, and it depends heavily on how
   common the disease is in whoever you point the model at.

### On the neural network

Same three-line shape as every other scikit-learn model - `create`, `.fit()`,
`.predict()` - wrapped in a `Pipeline` because it needs scaled inputs to train
well. On this dataset it usually ties the simpler models. That will not
always be true; it depends on how much data you have and how complicated the
true pattern is.

### The loop you ran across two days

Load data, split it (properly, with stratification), preprocess it (properly,
inside a pipeline), fit a model, cross-validate it, and only then - once,
right at the end - look at the locked-away test set. Everything else was one
more argument inside that same shape.

Thank you for two days. The notebook is yours to keep, break, and run again
from the top whenever you want.